In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abderhmanahmed/cbt-guide/therapists_guide_to_brief_cbtmanual.pdf
/kaggle/input/datasets/abderhmanahmed/therapsit-sound-test-1/TherapistAI_test1.mp3


In [2]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 65.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.6 MB/s eta 0:00:00:00:01
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_

In [3]:
!pip install transformers==4.52.4  langchain-classic

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

2026-04-29 20:08:53.747525: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777493334.134661      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777493334.252606      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777493335.284460      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777493335.284490      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777493335.284493      57 computation_placer.cc:177] computation placer alr

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer_ministrali= AutoTokenizer.from_pretrained(model_name)
model_minisralai = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [6]:
def generate_text(prompt, max_new_tokens=1000):
    inputs = tokenizer_ministrali(prompt, return_tensors="pt").to(model_minisralai.device)

    outputs = model_minisralai.generate(
        **inputs,
        max_new_tokens=max_new_tokens,   # ✅ only generate new text
        do_sample=False,                 # 🔥 deterministic
        temperature=0.0,                 # 🔥 no randomness
        pad_token_id=tokenizer_ministrali.eos_token_id
    )

    full_output = tokenizer_ministrali.decode(outputs[0], skip_special_tokens=True)

    # ✅ Remove prompt safely
    if full_output.startswith(prompt):
        generated = full_output[len(prompt):].strip()
    else:
        generated = full_output.strip()

    return generated

In [7]:
from langchain_core.language_models.llms import LLM
from typing import Any

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        return generate_text(prompt)
    
    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

llm = CustomHFLLM()

In [8]:
pdf_path = "/kaggle/input/datasets/abderhmanahmed/cbt-guide/therapists_guide_to_brief_cbtmanual.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)
vectordb = FAISS.from_documents(chunks, embedding)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import re

risklevel_schema= ResponseSchema(
    name="risk_level",
    description="A level to indicate the risk the user input is in from Low or medium or high."
)

selfharm_schema=ResponseSchema(
    name="self_harm",
    description="A boolean (ANSWER Only in[True or False]) which indicate weither the user input has intended to harm himself or already hurt himself or not"
)

suicidalintent_schema= ResponseSchema(
    name="suicidal_intent",
    description="A Boolean(ANSWER Only in[True or False]) to indicate weither the user input has the intended to suicide or not."
)

hopelessness_schema= ResponseSchema(
    name="hopelessness",
    description="A Boolean(ANSWER Only in[True or False]) to indicate weither the user input has hopelessness or not."
)

abuse_schema= ResponseSchema(
    name="abuse",
    description="A Boolean(ANSWER Only in[True or False]) to indicate weither the user is abused or not."
)
response_schemas=[risklevel_schema,selfharm_schema,suicidalintent_schema,hopelessness_schema,abuse_schema]
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

import json
safety_prompt = PromptTemplate(
    input_variables=["input","format_instructions"],
    template="""
You are a mental health safety classifier.

Respond ONLY in JSON format as follows,You MUST include ALL fields in the JSON.
Do not omit any keys:
{format_instructions}

User message:
{input}

Answer:
"""
)

safety_chain = LLMChain(
    llm=llm,
    prompt=safety_prompt
)

/tmp/ipykernel_57/1809728792.py:21: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  safety_chain = LLMChain(


In [11]:
cbt_prompt = PromptTemplate(
    input_variables=["input"],
    template="""
You are a CBT analyzer.

Extract ONLY JSON:
{{
  "emotion": "",
  "intensity": 0.0,
  "cognitive_distortion": "",
  "trigger": "",
  "automatic_thought": "",
  "core_belief": "",
  "behavior": ""
}}

User message:
{input}

Answer:
"""
)

cbt_chain = LLMChain(
    llm=llm,
    prompt=cbt_prompt
)

In [12]:
CBT_Rag_prompt= PromptTemplate(
    input_variables=["input", "cbt"],
    template="""
    You are a CBT (Cognitive Behavioral Therapy) summarization assistant.

Your task is to create a concise, structured summary that captures the psychological meaning of the conversation so it can be compared with CBT knowledge chunks.

Focus on:
- Core problem (situation)
- Emotions
- Negative thoughts / cognitive distortions
- Behaviors
- Key CBT concepts (if present)

Return ONLY a short structured summary in plain text.

---

CBT Data:
{cbt}

User Input:
{input}


Summary:
- Problem:
- Emotions:
- Thoughts:
- Behaviors:
- CBT Concepts:


Answer:
"""
)

CBT_Rag_chain = LLMChain(llm=llm, prompt=CBT_Rag_prompt)

In [13]:
def router(safety_result):
    risk = safety_result["risk_level"]
    #flags = safety_result["flags"]

    if risk == "High" or safety_result["self_harm"]=="True" or safety_result["suicidal_intent"]=="True":
        return "crisis"

    if risk == "Medium" or safety_result["hopelessness"]=="True" or safety_result["abuse"]=="True":
        return "support"

    return "full"

In [14]:
crisis_prompt = PromptTemplate(
    input_variables=["input","context","safety"],
    template="""
You are a supportive mental health assistant.

The user may be in distress.



User message:
{input}

Respond with empathy and encourage reaching out for help.
always add that the Egyption emergancy number is 122,Befrienders Cairo (Hotline): 7621602
General Secretariat of Mental Health: 08008880700 or 0220816831

Respond:
"""
)

crisis_chain = LLMChain(llm=llm, prompt=crisis_prompt)

In [15]:
support_prompt = PromptTemplate(
    input_variables=["input", "cbt","context"],
    template="""
You are a gentle CBT assistant.

Be supportive, do NOT challenge strongly.

CBT data:
{cbt}

User:
{input}

Use the following context to respond

Context:
{context}

Respond gently with validation + light reframing.

Respond:
"""
)

support_chain = LLMChain(llm=llm, prompt=support_prompt)

In [16]:
full_prompt = PromptTemplate(
    input_variables=["input", "cbt","context"],
    template="""
You are a CBT therapist assistant.

Use structured CBT:
- identify distortion
- challenge thought
- reframe belief
- suggest action



CBT data:
{cbt}

User:
{input}


Use the following context to respond

Context:
{context}

Respond:
"""
)

full_chain = LLMChain(llm=llm, prompt=full_prompt)

In [17]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [18]:
def run_pipeline(user_input):


    # 1. Safety
    safety_raw = safety_chain.run(
        input=user_input,
        format_instructions=format_instructions)
    safety = output_parser.parse(extract_json_block(safety_raw))

    # 2. CBT
    cbt_raw = cbt_chain.run(user_input)
    cbt = json.loads(cbt_raw)

    cbt_rag_result=CBT_Rag_chain.run(
        input=user_input,
        cbt=cbt_raw
    )
    
    docs = vectordb.similarity_search(cbt_rag_result, k=2)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    
    # 3. Route decision
    mode = router(safety)

    # 4. Select chain
    if mode == "crisis":
        response = crisis_chain.run(
            input=user_input,
            context=context
        )

    elif mode == "support":
        response = support_chain.run(
            input=user_input,
            cbt=cbt_raw,
            context=context
        )

    else:
        response = full_chain.run(
            input=user_input,
            cbt=cbt_raw,
            context=context
        )

    return {
        "mode": mode,
        "safety": safety,
        "cbt": cbt,
        "response": response
    }

In [19]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset


device = "cuda:1" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model_whisper = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model_whisper.to(device)

processor_whisper = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model_whisper,
    tokenizer=processor_whisper.tokenizer,
    feature_extractor=processor_whisper.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:1


In [20]:
result_audio=pipe("/kaggle/input/datasets/abderhmanahmed/therapsit-sound-test-1/TherapistAI_test1.mp3")
print(result_audio)

/usr/local/lib/python3.12/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


{'text': ' حزين زعلان اني عندي امتحان بس مبسوط اني اقرب تخلص'}


In [21]:
from huggingface_hub import hf_hub_download
import fasttext

model = fasttext.load_model(hf_hub_download("facebook/fasttext-language-identification", "model.bin"))

model.bin:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def translate(text, src_lang, tgt_lang="eng_Latn"):
    tokenizer.src_lang = src_lang
    
    inputs = tokenizer(text, return_tensors="pt")
    
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang)
    )
    
    return tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [23]:
import langid

lang, prob = langid.classify(result_audio["text"])
print(lang, prob)

ar -355.02484345436096


In [24]:
import langid

lang_map = {
    "en": "eng_Latn",
    "ar": "arb_Arab",
    "fr": "fra_Latn",
    "de": "deu_Latn",
    "es": "spa_Latn"
}


def translate_pipeline(text, target_lang="eng_Latn"):
    lang, prob = langid.classify(text)
    
    if lang not in lang_map:
        return f"Unsupported language: {lang}"
    
    src_lang = lang_map[lang]
    
    translation = translate(text, src_lang, target_lang)
    
    return {
        "detected_language": lang,
        "confidence": prob,
        "translation": translation
    }

print(translate_pipeline(result_audio["text"]))

{'detected_language': 'ar', 'confidence': -355.02484345436096, 'translation': "Sad to say I have an exam, but I'm glad I'm close."}


In [25]:
'''#esult_audio=pipe("/kaggle/input/datasets/abderhmanahmed/therapsit-sound-test-1/TherapistAI_test1.mp3")

userInput=translate_pipeline("Yes")

result=run_pipeline(userInput["translation"])

detected_lang = userInput["detected_language"]

if detected_lang in lang_map and detected_lang != "en":
    print(
        translate(
            result["response"],
            "eng_Latn",
           lang_map[detected_lang]))
else:
    print(result["response"])'''

'#esult_audio=pipe("/kaggle/input/datasets/abderhmanahmed/therapsit-sound-test-1/TherapistAI_test1.mp3")\n\nuserInput=translate_pipeline("Yes")\n\nresult=run_pipeline(userInput["translation"])\n\ndetected_lang = userInput["detected_language"]\n\nif detected_lang in lang_map and detected_lang != "en":\n    print(\n        translate(\n            result["response"],\n            "eng_Latn",\n           lang_map[detected_lang]))\nelse:\n    print(result["response"])'

In [26]:
pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q

Note: you may need to restart the kernel to use updated packages.


In [27]:
from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel
import tempfile

app = FastAPI()

class TextRequest(BaseModel):
    message: str


# =========================
# SAFE TEXT PIPELINE
# =========================
def process_input(user_input_text):
    try:
        userInput = translate_pipeline(user_input_text)
        result = run_pipeline(userInput["translation"])

        detected_lang = userInput["detected_language"]

        response = result.get("response", "")

        if detected_lang in lang_map and detected_lang != "en":
            response = translate(
                response,
                "eng_Latn",
                lang_map[detected_lang]
            )

        return response

    except Exception as e:
        return f"❌ Text pipeline error: {str(e)}"


# =========================
# CHAT ENDPOINT
# =========================
@app.post("/chat")
async def chat_text(req: TextRequest):
    response = process_input(req.message)

    return {
        "response": response
    }


# =========================
# VOICE ENDPOINT (FIXED)
# =========================
@app.post("/voice")
@app.post("/voice")
async def chat_audio(file: UploadFile = File(...)):
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
            tmp.write(await file.read())
            path = tmp.name

        result_audio = pipe(path)

        if not result_audio or "text" not in result_audio:
            return {
                "response": "❌ Could not transcribe audio",
                "transcription": ""
            }

        text = result_audio["text"]

        response = process_input(text)

        return {
            "transcription": text,
            "response": response
        }

    except Exception as e:
        return {
            "response": f"❌ Voice error: {str(e)}",
            "transcription": ""
        }

In [28]:
NGROK_TOKEN = "3D0njpSvOZQ77NRwUKHH1YysX8I_6Aifeyu7Euqvwcb3DPxPD"
API_KEY = "secret123"

In [29]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://aversion-dicing-credibly.ngrok-free.dev                                    


INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:51623 (Press CTRL+C to quit)


In [30]:
userInput="I am stressed because of academic preesure and can't sleep well.".

result=run_pipeline(userInput)
print(result)

'userInput="I am ."\n\nresult=run_pipeline(userInput)\nprint(result)'

In [31]:
#print(result["response"])